# Time-Evolution Results Computation

This notebook runs the selected pairing-quench time-evolution experiments from `FYS5429Lipkin.ipynb` using the shared helper modules. It saves raw arrays, predictions, summary tables, and settings only. It does not create plots.



This notebook was adapted from original time evolution cells written by myself. Codex and ChatGPT helped build the experiment skeleton around the shared time_evolution.py helpers.


In [1]:

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from IPython.display import display

from common import GLOBAL_SEED
from baselines import run_gru_grid_search, train_final_gru_from_config
from time_evolution import (
    make_pairing_quench_observable_data,
    train_pmm_survival_model,
    train_pmm_observable_model,
    train_pmm_fixed_observable_model,
    build_exact_quench_compression_basis,
    evaluate_compressed_exact_quench_by_rank,
    compact_model_result,
    comparison_row,
    compressed_exact_comparison_row,
    matched_pmm_compressed_error_rows,
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
TIME_EVOLUTION_RESULTS_PATH = RESULTS_DIR / "time_evolution_results.pkl"
TIME_EVOLUTION_SCHEMA_VERSION = "compressed_exact_time_evolution_v2"


def save_results(results, path=TIME_EVOLUTION_RESULTS_PATH):
    """Save the current time evolution result dict to a pickle path and return None."""
    with path.open("wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved time-evolution results to {path}")


def compact_data_for_save(data):
    """Select serializable arrays from a quench data dict and return a smaller dict."""
    keys = [
        "observable_name",
        "g_prep",
        "g_train",
        "t_train",
        "g_test",
        "t_test",
        "Y_train",
        "Y_test",
        "X_train_seq",
        "X_test_seq",
        "X_train_pmm",
        "X_test_pmm",
        "y_train_flat",
        "y_test_flat",
        "g_min",
        "g_max",
        "t_min",
        "t_max",
        "x_prep",
    ]
    return {key: data[key] for key in keys}


def compact_pmm_result(result, data):
    """Drop torch model state from a PMM result and add grid shaped predictions."""
    compact = compact_model_result(result)
    compact["Y_test_pred"] = np.asarray(compact["y_test_pred_flat"]).reshape(data["Y_test"].shape)
    compact["Y_test_pred_seq"] = compact["Y_test_pred"]
    compact["abs_error"] = np.abs(compact["Y_test_pred"] - data["Y_test"])
    return compact


def compact_gru_result(result, data):
    """Drop torch model state from a GRU result and add grid shaped predictions."""
    compact = compact_model_result(result)
    compact["Y_test_pred"] = np.asarray(compact["Y_test_pred_seq"])
    compact["abs_error"] = np.abs(compact["Y_test_pred"] - data["Y_test"])
    return compact


def run_time_evolution_observable(observable_name, settings, compression_settings, pmm_settings, gru_settings, compression_basis=None):
    """Run exact compression, PMM, and GRU experiments for one observable and return a result block.

    This experiment skeleton was organized with Codex and ChatGPT from the
    original time evolution notebook cells and shared helper code.
    """
    data = make_pairing_quench_observable_data(
        pnum=settings["pnum"],
        hnum=settings["hnum"],
        delta=settings["delta"],
        g_prep=settings["g_prep"],
        g_min=settings["g_min"],
        g_max=settings["g_max"],
        t_min=settings["t_min"],
        t_max=settings["t_max"],
        n_g_train=settings["n_g_train"],
        n_t_train=settings["n_t_train"],
        n_g_test=settings["n_g_test"],
        n_t_test=settings["n_t_test"],
        observable_name=observable_name,
    )

    if compression_basis is None:
        compression_basis = build_exact_quench_compression_basis(
            data["pairing"],
            k=compression_settings["k"],
            g_min=settings["g_min"],
            g_max=settings["g_max"],
            ranks=compression_settings["ranks"],
            n_grid=compression_settings["q_grid"],
        )

    compressed_exact_by_rank = evaluate_compressed_exact_quench_by_rank(
        data,
        compression_basis,
        compression_settings["ranks"],
    )

    pmm_by_n = {}
    pmm_fixed_observable_by_n = {}
    comparison_rows = []
    for rank, result in sorted(compressed_exact_by_rank.items()):
        comparison_rows.append(compressed_exact_comparison_row(f"Compressed exact r={rank}", result, rank=rank))

    is_survival = observable_name == "survival"
    train_pmm = train_pmm_survival_model if is_survival else train_pmm_observable_model

    for n_pmm in pmm_settings["n_pmm_values"]:
        print("\n" + "=" * 72)
        print(f"Training PMM for observable={observable_name}, n={n_pmm}, epochs={pmm_settings['epochs']}")
        print("=" * 72)
        result = train_pmm(
            data["X_train_pmm"],
            data["y_train_flat"],
            data["X_test_pmm"],
            data["y_test_flat"],
            n_pmm=n_pmm,
            x_prep=data["x_prep"],
            epochs=pmm_settings["epochs"],
            lr=pmm_settings["learning_rate"],
            init_scale=pmm_settings["init_scale"],
            eval_batch_size=pmm_settings.get("eval_batch_size"),
            seed=pmm_settings["seed"],
            verbose=pmm_settings["verbose"],
            print_every=pmm_settings["print_every"],
        )
        pmm_by_n[int(n_pmm)] = compact_pmm_result(result, data)
        pmm_family = "PMM_survival" if is_survival else "PMM_learned_observable"
        comparison_rows.append(comparison_row(f"PMM n={n_pmm}", pmm_family, result, n_pmm=n_pmm))

        if (not is_survival) and pmm_settings.get("run_fixed_observable_pmm", True):
            rank = int(n_pmm)
            O_reduced = compressed_exact_by_rank[rank]["O_reduced"]
            print("\n" + "=" * 72)
            print(f"Training fixed-observable PMM for observable={observable_name}, n={n_pmm}")
            print("=" * 72)
            fixed_result = train_pmm_fixed_observable_model(
                data["X_train_pmm"],
                data["y_train_flat"],
                data["X_test_pmm"],
                data["y_test_flat"],
                O_fixed=O_reduced,
                x_prep=data["x_prep"],
                epochs=pmm_settings["epochs"],
                lr=pmm_settings["learning_rate"],
                init_scale=pmm_settings["init_scale"],
                eval_batch_size=pmm_settings.get("eval_batch_size"),
                seed=pmm_settings["seed"],
                verbose=pmm_settings["verbose"],
                print_every=pmm_settings["print_every"],
            )
            pmm_fixed_observable_by_n[rank] = compact_pmm_result(fixed_result, data)
            comparison_rows.append(comparison_row(f"PMM fixed O n={n_pmm}", "PMM_fixed_observable", fixed_result, n_pmm=n_pmm))

    print("\n" + "=" * 72)
    print(f"Running GRU grid search for observable={observable_name}, epochs={gru_settings['epochs']}")
    print("=" * 72)
    gru_grid_summary, best_gru_config, split_indices = run_gru_grid_search(
        data["X_train_seq"],
        data["Y_train"],
        hidden_dims=tuple(gru_settings["hidden_dims"]),
        num_layers_options=tuple(gru_settings["num_layers_options"]),
        dropouts=tuple(gru_settings["dropouts"]),
        lr=gru_settings["learning_rate"],
        epochs=gru_settings["epochs"],
        val_fraction=gru_settings["validation_fraction"],
        seed=gru_settings["seed"],
        verbose=gru_settings["verbose"],
    )

    print("\n" + "=" * 72)
    print(f"Training final GRU for observable={observable_name} with best config")
    print("=" * 72)
    gru_final = train_final_gru_from_config(
        data["X_train_seq"],
        data["Y_train"],
        data["X_test_seq"],
        data["Y_test"],
        best_gru_config,
        lr=gru_settings["learning_rate"],
        epochs=gru_settings["epochs"],
        seed=gru_settings["seed"],
        verbose=gru_settings["verbose"],
    )
    comparison_rows.append(comparison_row("GRU best", "GRU", gru_final))

    comparison_table = pd.DataFrame(comparison_rows).sort_values("mean_abs_test_error").reset_index(drop=True)
    matched_rows = matched_pmm_compressed_error_rows(
        pmm_by_n,
        compressed_exact_by_rank,
        model_family="PMM_survival" if is_survival else "PMM_learned_observable",
    )
    if pmm_fixed_observable_by_n:
        matched_rows.extend(
            matched_pmm_compressed_error_rows(
                pmm_fixed_observable_by_n,
                compressed_exact_by_rank,
                model_family="PMM_fixed_observable",
            )
        )
    matched_compressed_table = pd.DataFrame(matched_rows)
    compressed_exact_summary = pd.DataFrame(
        [row for row in comparison_rows if row["model_family"] == "compressed_exact"]
    ).sort_values("rank").reset_index(drop=True)
    best_pmm_n = int(
        comparison_table[comparison_table["model_family"].str.startswith("PMM")]
        .sort_values("mean_abs_test_error")
        .iloc[0]["n_pmm"]
    )

    return {
        "settings": {
            "observable_name": observable_name,
            "data": settings,
            "compression": compression_settings,
            "pmm": pmm_settings,
            "gru": gru_settings,
        },
        "data": compact_data_for_save(data),
        "compression_basis": compression_basis,
        "compressed_exact_by_rank": compressed_exact_by_rank,
        "compressed_exact_summary": compressed_exact_summary,
        "matched_pmm_compressed_table": matched_compressed_table,
        "comparison_table": comparison_table,
        "pmm_by_n": pmm_by_n,
        "pmm_fixed_observable_by_n": pmm_fixed_observable_by_n,
        "best_pmm_n": best_pmm_n,
        "gru_grid_summary": gru_grid_summary,
        "gru_best_config": best_gru_config,
        "gru_final": compact_gru_result(gru_final, data),
        "train_validation_indices": split_indices,
    }


def run_compression_k_sensitivity(observable_name, settings, compression_settings):
    """Evaluate compressed exact dynamics for several k values and return summary data."""
    data = make_pairing_quench_observable_data(
        pnum=settings["pnum"],
        hnum=settings["hnum"],
        delta=settings["delta"],
        g_prep=settings["g_prep"],
        g_min=settings["g_min"],
        g_max=settings["g_max"],
        t_min=settings["t_min"],
        t_max=settings["t_max"],
        n_g_train=settings["n_g_train"],
        n_t_train=settings["n_t_train"],
        n_g_test=settings["n_g_test"],
        n_t_test=settings["n_t_test"],
        observable_name=observable_name,
    )
    rows = []
    blocks_by_k = {}
    for k in compression_settings.get("k_sensitivity_values", [compression_settings["k"]]):
        print("\n" + "=" * 72)
        print(f"Compressed exact k-sensitivity for observable={observable_name}, k={k}")
        print("=" * 72)
        basis = build_exact_quench_compression_basis(
            data["pairing"],
            k=k,
            g_min=settings["g_min"],
            g_max=settings["g_max"],
            ranks=compression_settings["ranks"],
            n_grid=compression_settings["q_grid"],
        )
        compressed = evaluate_compressed_exact_quench_by_rank(
            data,
            basis,
            compression_settings["ranks"],
        )
        for rank, result in sorted(compressed.items()):
            row = compressed_exact_comparison_row(f"Compressed exact k={k}, r={rank}", result, rank=rank)
            row["k"] = int(k)
            row["observable"] = observable_name
            rows.append(row)
        blocks_by_k[int(k)] = {
            "compression_basis": basis,
            "compressed_exact_by_rank": compressed,
        }

    summary = pd.DataFrame(rows).sort_values(["k", "rank"]).reset_index(drop=True)
    return {
        "settings": {
            "observable_name": observable_name,
            "data": settings,
            "compression": compression_settings,
        },
        "data": compact_data_for_save(data),
        "summary": summary,
        "blocks_by_k": blocks_by_k,
    }


time_evolution_results = {}


## Shared Settings

The quench setup follows the new time-evolution cells in the original notebook. The exact-compression ranks are matched to the PMM dimensions, and survival compression is also checked for several Q_k values. Reduced exact dynamics uses the projected initial state `B_r.T @ psi0`, renormalized before time evolution; the saved projection norm is therefore an initial-state capture diagnostic. PMM dimensions all use 4000 epochs. The GRU baseline performs a wider hidden-width/layer grid and trains each configuration for 10000 epochs.


In [2]:

quench_settings = {
    "pnum": 8,
    "hnum": 8,
    "delta": 1.0,
    "g_prep": 0.5,
    "g_min": 0.0,
    "g_max": 2.0,
    "n_g_train": 24,
    "n_t_train": 32,
    "n_g_test": 48,
    "n_t_test": 80,
    "t_min": 0.0,
    "t_max": 12.0,
    "plot_level_g_values": [0.25, 1.0, 1.75],
}

compression_time_settings = {
    "k": 3,
    "k_sensitivity_values": [3, 5, 8],
    "ranks": [16, 24, 32],
    "q_grid": 80,
}

pmm_time_settings = {
    "n_pmm_values": [16, 24, 32],
    "epochs": 4000,
    "learning_rate": 1e-2,
    "init_scale": 1e-2,
    "eval_batch_size": 512,
    "seed": GLOBAL_SEED,
    "verbose": False,
    "print_every": 500,
    "run_fixed_observable_pmm": True,
}

gru_time_settings = {
    "hidden_dims": [32, 64, 128],
    "num_layers_options": [1, 2, 3],
    "dropouts": [0.0],
    "epochs": 10000,
    "learning_rate": 1e-3,
    "validation_fraction": 0.25,
    "seed": GLOBAL_SEED,
    "verbose": False,
}

time_evolution_results["settings"] = {
    "quench": quench_settings,
    "compression": compression_time_settings,
    "pmm": pmm_time_settings,
    "gru": gru_time_settings,
}


## Compression k Sensitivity

Calibrates the exact compressed survival dynamics for several Q_k construction choices before using the main k value in the PMM comparisons.


In [3]:

survival_compression_k_sensitivity = run_compression_k_sensitivity(
    "survival",
    settings=quench_settings,
    compression_settings=compression_time_settings,
)
time_evolution_results["compression_k_sensitivity"] = survival_compression_k_sensitivity
save_results(time_evolution_results)
survival_compression_k_sensitivity["summary"]



Compressed exact k-sensitivity for observable=survival, k=3

Compressed exact k-sensitivity for observable=survival, k=5

Compressed exact k-sensitivity for observable=survival, k=8
Saved time-evolution results to results/time_evolution_results.pkl


,model,model_family,n_pmm,rank,effective_dimension,mean_abs_test_error,max_abs_test_error,rms_test_error,test_mse,parameter_count,train_time_s,best_epoch,best_train_mse,psi0_projected_norm,psi0_projected_weight,k,observable
0,"Compressed exact k=3, r=16",compressed_exact,NaN,16,16,0.012368,0.097747,0.019896,0.000396,NaN,0.0,NaN,NaN,1.000000,1.000000,3,survival
1,"Compressed exact k=3, r=24",compressed_exact,NaN,24,24,0.013274,0.106798,0.021163,0.000448,NaN,0.0,NaN,NaN,1.000000,1.000000,3,survival
2,"Compressed exact k=3, r=32",compressed_exact,NaN,32,32,0.007086,0.075886,0.012921,0.000167,NaN,0.0,NaN,NaN,1.000000,1.000000,3,survival
3,"Compressed exact k=5, r=16",compressed_exact,NaN,16,16,0.005521,0.039883,0.008374,0.000070,NaN,0.0,NaN,NaN,0.999999,0.999997,5,survival
4,"Compressed exact k=5, r=24",compressed_exact,NaN,24,24,0.004861,0.038413,0.007463,0.000056,NaN,0.0,NaN,NaN,1.000000,1.000000,5,survival
5,"Compressed exact k=5, r=32",compressed_exact,NaN,32,32,0.003816,0.037300,0.006199,0.000038,NaN,0.0,NaN,NaN,1.000000,1.000000,5,survival
6,"Compressed exact k=8, r=16",compressed_exact,NaN,16,16,0.013745,0.321193,0.034842,0.001214,NaN,0.0,NaN,NaN,0.999878,0.999756,8,survival
7,"Compressed exact k=8, r=24",compressed_exact,NaN,24,24,0.002435,0.028206,0.004325,0.000019,NaN,0.0,NaN,NaN,0.999989,0.999979,8,survival
8,"Compressed exact k=8, r=32",compressed_exact,NaN,32,32,0.002045,0.026233,0.003739,0.000014,NaN,0.0,NaN,NaN,1.000000,1.000000,8,survival


## 1. Survival Probability

Runs full exact survival data, exact compressed survival at matched ranks, PMM survival models at matched dimensions, and a GRU grid search.


In [4]:

survival_results = run_time_evolution_observable(
    "survival",
    settings=quench_settings,
    compression_settings=compression_time_settings,
    pmm_settings=pmm_time_settings,
    gru_settings=gru_time_settings,
)
time_evolution_results["compression_basis"] = survival_results["compression_basis"]
survival_results = dict(survival_results)
survival_results.pop("compression_basis", None)
time_evolution_results["survival"] = survival_results

combined_rows = [survival_results["comparison_table"].assign(observable="survival")]
time_evolution_results["combined_summary"] = pd.concat(combined_rows, ignore_index=True)
time_evolution_results["matched_pmm_compressed_summary"] = survival_results["matched_pmm_compressed_table"].assign(observable="survival")

save_results(time_evolution_results)
survival_results["comparison_table"]



Training PMM for observable=survival, n=16, epochs=4000

Training PMM for observable=survival, n=24, epochs=4000

Training PMM for observable=survival, n=32, epochs=4000

Running GRU grid search for observable=survival, epochs=10000

Training final GRU for observable=survival with best config
Saved time-evolution results to results/time_evolution_results.pkl


,model,model_family,n_pmm,rank,effective_dimension,mean_abs_test_error,max_abs_test_error,rms_test_error,test_mse,parameter_count,train_time_s,best_epoch,best_train_mse,psi0_projected_norm,psi0_projected_weight
0,Compressed exact r=32,compressed_exact,NaN,32.0,32.0,0.007086,0.075886,0.012921,0.000167,NaN,0.000000,NaN,NaN,1.0,1.0
1,Compressed exact r=16,compressed_exact,NaN,16.0,16.0,0.012368,0.097747,0.019896,0.000396,NaN,0.000000,NaN,NaN,1.0,1.0
2,PMM n=16,PMM_survival,16.0,NaN,16.0,0.012792,0.100365,0.018239,0.000333,512.0,52.881745,3974.0,0.000341,NaN,NaN
3,Compressed exact r=24,compressed_exact,NaN,24.0,24.0,0.013274,0.106798,0.021163,0.000448,NaN,0.000000,NaN,NaN,1.0,1.0
4,PMM n=32,PMM_survival,32.0,NaN,32.0,0.021458,0.143976,0.032730,0.001071,2048.0,168.854073,3949.0,0.001003,NaN,NaN
5,PMM n=24,PMM_survival,24.0,NaN,24.0,0.021510,0.149161,0.032260,0.001041,1152.0,101.338767,3948.0,0.001026,NaN,NaN
6,GRU best,GRU,NaN,NaN,NaN,0.123444,0.794667,0.186738,0.034871,50817.0,87.859069,9908.0,0.000005,NaN,NaN


## 2. Fixed Pair-Occupation Observable

Repeats the exact-compression/PMM/GRU comparison for the lowest-level pair occupation. This block saves both the learned-observable PMM and the fixed compressed-observable PMM separately. The fixed-observable PMM is dimension-matched to the compressed model, but its latent basis is not canonically basis-matched to the exact compressed basis.


In [6]:

observable_names = ["pair_occ_0"]
observable_blocks = {}
summary_blocks = []
matched_summary_blocks = []

for observable_name in observable_names:
    block = run_time_evolution_observable(
        observable_name,
        settings=quench_settings,
        compression_settings=compression_time_settings,
        pmm_settings=pmm_time_settings,
        gru_settings=gru_time_settings,
        compression_basis=time_evolution_results["compression_basis"],
    )
    block.pop("compression_basis", None)
    observable_blocks[observable_name] = block
    summary_blocks.append(block["comparison_table"].assign(observable=observable_name))
    matched_summary_blocks.append(block["matched_pmm_compressed_table"].assign(observable=observable_name))

observable_summary = pd.concat(summary_blocks, ignore_index=True)
observable_matched_summary = pd.concat(matched_summary_blocks, ignore_index=True)
time_evolution_results["observables"] = {
    "settings": {
        "observable_names": observable_names,
        "data": quench_settings,
        "compression": compression_time_settings,
        "pmm": pmm_time_settings,
        "gru": gru_time_settings,
    },
    "blocks": observable_blocks,
    "summary": observable_summary,
    "matched_pmm_compressed_summary": observable_matched_summary,
}

time_evolution_results["combined_summary"] = pd.concat(
    [survival_results["comparison_table"].assign(observable="survival"), observable_summary],
    ignore_index=True,
)
time_evolution_results["matched_pmm_compressed_summary"] = pd.concat(
    [survival_results["matched_pmm_compressed_table"].assign(observable="survival"), observable_matched_summary],
    ignore_index=True,
)

save_results(time_evolution_results)
observable_summary



Training PMM for observable=pair_occ_0, n=16, epochs=4000

Training fixed-observable PMM for observable=pair_occ_0, n=16

Training PMM for observable=pair_occ_0, n=24, epochs=4000

Training fixed-observable PMM for observable=pair_occ_0, n=24

Training PMM for observable=pair_occ_0, n=32, epochs=4000

Training fixed-observable PMM for observable=pair_occ_0, n=32

Running GRU grid search for observable=pair_occ_0, epochs=10000

Training final GRU for observable=pair_occ_0 with best config
Saved time-evolution results to results/time_evolution_results.pkl


,model,model_family,n_pmm,rank,effective_dimension,mean_abs_test_error,max_abs_test_error,rms_test_error,test_mse,parameter_count,train_time_s,best_epoch,best_train_mse,psi0_projected_norm,psi0_projected_weight,observable
0,Compressed exact r=32,compressed_exact,NaN,32.0,32.0,0.008299,0.069974,0.014350,0.000206,NaN,0.000000,NaN,NaN,1.0,1.0,pair_occ_0
1,PMM fixed O n=32,PMM_fixed_observable,32.0,NaN,32.0,0.008890,0.094097,0.013966,0.000195,2048.0,249.588275,4000.0,0.000175,NaN,NaN,pair_occ_0
2,Compressed exact r=24,compressed_exact,NaN,24.0,24.0,0.010476,0.088729,0.018108,0.000328,NaN,0.000000,NaN,NaN,1.0,1.0,pair_occ_0
3,PMM n=32,PMM_learned_observable,32.0,NaN,32.0,0.010564,0.127917,0.017846,0.000318,3072.0,230.401292,4000.0,0.000337,NaN,NaN,pair_occ_0
4,PMM n=24,PMM_learned_observable,24.0,NaN,24.0,0.010577,0.128039,0.017901,0.000320,1728.0,146.334237,4000.0,0.000340,NaN,NaN,pair_occ_0
5,Compressed exact r=16,compressed_exact,NaN,16.0,16.0,0.010625,0.085021,0.018311,0.000335,NaN,0.000000,NaN,NaN,1.0,1.0,pair_occ_0
6,PMM fixed O n=16,PMM_fixed_observable,16.0,NaN,16.0,0.011115,0.132509,0.018234,0.000332,512.0,66.821341,4000.0,0.000352,NaN,NaN,pair_occ_0
7,PMM fixed O n=24,PMM_fixed_observable,24.0,NaN,24.0,0.011393,0.136311,0.018466,0.000341,1152.0,152.921689,4000.0,0.000363,NaN,NaN,pair_occ_0
8,PMM n=16,PMM_learned_observable,16.0,NaN,16.0,0.011519,0.139266,0.018860,0.000356,768.0,67.808146,3885.0,0.000378,NaN,NaN,pair_occ_0
9,GRU best,GRU,NaN,NaN,NaN,0.043440,0.230319,0.050024,0.002502,50817.0,98.267191,10000.0,0.000007,NaN,NaN,pair_occ_0
